In [1]:
import boto3

bucket_name = "dataminds-warehouse"
s3_file_key = "ramen-ratings.csv"  # e.g. 'folder/myfile.txt'
local_file_path = "ramen-ratings.csv"  # Local destination

# Create an S3 client (remove `bucket_name` here — not a valid argument for boto3.client)
s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    # aws_access_key_id='your_access_key',
    # aws_secret_access_key='your_secret_key'
)

# Download the file
try:
    s3.download_file(bucket_name, s3_file_key, local_file_path)
    print(
        f"✅ File downloaded successfully from s3://{bucket_name}/{s3_file_key} to {local_file_path}"
    )
except Exception as e:
    print("❌ Error downloading file:", e)

✅ File downloaded successfully from s3://dataminds-warehouse/ramen-ratings.csv to ramen-ratings.csv


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score
from category_encoders import CatBoostEncoder
from sentence_transformers import SentenceTransformer

In [3]:
# %pip install sentence_transformers

Note: you may need to restart the kernel to use updated packages.


In [4]:
# %pip install category_encoders

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached category_encoders-2.8.1-py3-none-any.whl.metadata (7.9 kB)
Using cached category_encoders-2.8.1-py3-none-any.whl (85 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
class AddEmbed(BaseEstimator, TransformerMixin):
    def __init__(self, embedding_col="variety", n_components=100):
        self.embedding_col = embedding_col
        self.n_components = n_components
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.pca = PCA(n_components=n_components, random_state=42)

    def fit(self, X, y=None):
        texts = X[self.embedding_col].astype(str).tolist()
        embs = self.model.encode(texts, show_progress_bar=False)
        self.pca.fit(embs)
        return self

    def transform(self, X, y=None):
        texts = X[self.embedding_col].astype(str).tolist()
        embs = self.model.encode(texts, show_progress_bar=False)
        embs_pca = self.pca.transform(embs)
        emb_df = pd.DataFrame(
            embs_pca,
            index=X.index,
            columns=[f"{self.embedding_col}_emb_{i}" for i in range(1, embs_pca.shape[1] + 1)],
        )
        return pd.concat([X.drop(columns=[self.embedding_col]), emb_df], axis=1)

In [4]:
class RamenPipeline:
    def __init__(self, target="stars", task="auto"):
        self.target = target
        self.task = task  # "auto", "regression", "classification"
        self.model = None
        self.pipeline = None

    def load_and_clean(self, file_path):
        df = pd.read_csv(file_path)
        df.set_index("Review #", inplace=True)

        df.columns = df.columns.str.lower()

        df[self.target] = df[self.target].astype(str).str.strip().str.lower()
        df = df[df[self.target] != "unrated"]

        # Convert stars to float
        df[self.target] = df[self.target].astype(float)

        if "top ten" in df.columns:
            df = df.drop(columns=["top ten"])
        return df

    def build_pipeline(self, X):
        categorical_features = ["brand", "style", "country"]
        numeric_features = X.select_dtypes(include=["float32", "float64", "int64"]).columns.tolist()

        numeric_transformer = Pipeline(
            steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler())]
        )

        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", CatBoostEncoder()),
            ]
        )

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numeric_transformer, numeric_features),
                ("cat", categorical_transformer, categorical_features),
            ],
            remainder="passthrough",
        )

        self.pipeline = Pipeline(
            steps=[
                ("embed", AddEmbed(embedding_col="variety", n_components=100)),
                ("preprocessor", preprocessor),
            ]
        )

    def fit(self, df):
        X = df.drop(columns=[self.target])
        y = df[self.target]

        # Auto-detecting the task
        if self.task == "auto":
            if y.nunique() <= 10 and all(y.astype(int) == y):  # looks categorical
                self.task = "classification"
            else:
                self.task = "regression"

        self.build_pipeline(X)
        X_proc = self.pipeline.fit_transform(X, y)

        if self.task == "classification":
            self.model = RandomForestClassifier(max_depth=10, n_estimators=300, random_state=42)
        else:
            self.model = RandomForestRegressor(max_depth=10, n_estimators=300, random_state=42)

        self.model.fit(X_proc, y)
        return X_proc, y

    def evaluate(self, df):
        X = df.drop(columns=[self.target])
        y = df[self.target]
        X_proc = self.pipeline.transform(X)
        y_pred = self.model.predict(X_proc)

        if self.task == "classification":
            score = accuracy_score(y, y_pred.round())
            print(f"Accuracy: {score:.3f}")
        else:
            mse = mean_squared_error(y, y_pred)
            print(f"MSE: {mse:.3f}")

        print(f"Transformed Test Shape: {X_proc.shape}")

    def run(self, file_path, test_size=0.2):
        df = self.load_and_clean(file_path)
        X_train, X_test = train_test_split(df, test_size=test_size, random_state=42)

        print("\nTraining...")
        self.fit(X_train)

        print("\nEvaluating on Test Set...")
        self.evaluate(X_test)

In [7]:
pipeline = RamenPipeline(task="auto")
pipeline.run("ramen-ratings.csv")


Training...

Evaluating on Test Set...
MSE: 0.830
Transformed Test Shape: (516, 103)


In [9]:
"""The SBERT embeddings (varietal) added a lot to the feature space(sensorily, these are how I understand ramen product names).

CatBoost encoding outperformed one-hot encoding in dealing with the high-cardinality brand feature.

Skew in categorical distributions (e.g. majority of mn_codes =“pack”) had no effect on RandomForestRegressor because it ignored high-cardinality categories anyway

PCA control to clipped embedding size, that means dimensionality was reduced while preserving information.
"""

'The SBERT embeddings (varietal) added a lot to the feature space(sensorily, these are how I understand ramen product names).\n\nCatBoost encoding outperformed one-hot encoding in dealing with the high-cardinality brand feature.\n\nSkew in categorical distributions (e.g. majority of mn_codes =“pack”) had no effect on RandomForestRegressor because it ignored high-cardinality categories anyway\n\nPCA control to clipped embedding size, that means dimensionality was reduced while preserving information.'

In [ ]:
import boto3

# Replace with your actual credentials and info
bucket_name = "dataminds-homeworks"
s3_file_key = "ali-gasimov-fe2.ipynb"
local_file_path = "ali-gasimov-fe2.ipynb"

# Create an S3 client
s3 = boto3.client("s3")

# Upload the file
try:
    s3.upload_file(local_file_path, bucket_name, s3_file_key)
    print(f"File uploaded successfully to s3://{bucket_name}/{s3_file_key}")
except Exception as e:
    print("Error uploading file:", e)